Deliverable: unified, cleaned dataset

In [122]:
import pandas as pd
import numpy as np
import re
import html

In [123]:
from pathlib import Path
import sys

# Get the current working directory
PROJECT_ROOT = Path.cwd()

# If running from the notebooks folder, go up one level
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python's import path
sys.path.insert(0, str(PROJECT_ROOT))

# Import loader
from src.loader import load_csv_folder

# Dataset directory
ZENODO_DIR = PROJECT_ROOT / "data" / "raw" / "zenodo"

# put all csv files into single df
data_df = load_csv_folder(ZENODO_DIR)

#display data
data_df

Loading CEAS_08.csv...
Loaded 39154 rows
Loading Enron.csv...
Loaded 29767 rows
Loading Ling.csv...
Loaded 2859 rows
Loading Nazario.csv...
Loaded 1565 rows
Loading Nazario_5.csv...
Loaded 3065 rows
Loading Nigerian_5.csv...
Loaded 6331 rows
Loading Nigerian_Fraud.csv...
Loaded 3332 rows
Loading SpamAssasin.csv...
Loaded 5809 rows
Loading TREC_05.csv...
Loaded 55990 rows
Loading TREC_06.csv...
Loaded 16457 rows
Loading TREC_07.csv...
Loaded 53757 rows

All datasets combined
Total rows: 218086
Total columns: 8


,sender,receiver,date,subject,body,label,urls,source_file
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1.0,1,CEAS_08.csv
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1.0,1,CEAS_08.csv
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1.0,1,CEAS_08.csv
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0.0,1,CEAS_08.csv
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1.0,1,CEAS_08.csv
...,...,...,...,...,...,...,...,...
218081,SCC <Gerry.Rossi4360@kinki-kids.com>,Deficient <deficient@flax9.uwaterloo.ca>,"Fri, 06 Jul 2007 06:53:36 -0400",Job: just for you.,\n\n\n\nWhile we may have high ...,1.0,1,TREC_07.csv
218082,Sydney Car Centre <Merrill8783@168city.com>,Gnitpick <gnitpick@flax9.uwaterloo.ca>,"Fri, 06 Jul 2007 06:59:51 -0400",the reply for your request for a job place [le...,\n\n\n\nWhile we may have high ...,1.0,1,TREC_07.csv
218083,Philippe Grosjean <phgrosjean@sciviews.org>,Duncan Murdoch <murdoch@stats.uwo.ca>,"Fri, 06 Jul 2007 12:57:17 +0200","Re: [R] Me again, about the horrible documenta...","For those who are interested, I just cook a li...",0.0,1,TREC_07.csv
218084,Bernhard Wellhöfer <Bernhard.Wellhoefer@gaia-g...,r-help@stat.math.ethz.ch,"Fri, 06 Jul 2007 12:43:12 +0200",Re: [R] RODBC problem,"Hello,\n\nas I wrote I call\n\n sqlFetch(chan...",0.0,1,TREC_07.csv


In [124]:
print(data_df.columns.tolist())
print(data_df.info())

['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls', 'source_file']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218086 entries, 0 to 218085
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   sender       184505 non-null  object 
 1   receiver     178821 non-null  object 
 2   date         182172 non-null  object 
 3   subject      214861 non-null  object 
 4   body         217466 non-null  object 
 5   label        217314 non-null  float64
 6   urls         184688 non-null  object 
 7   source_file  218086 non-null  object 
dtypes: float64(1), object(7)
memory usage: 13.3+ MB
None


In [125]:
# stamdardize case and remove whitespace from column names
data_df.columns = (
    data_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [126]:
#missing values
print(data_df[["subject", "body"]].isna().sum())

subject    3225
body        620
dtype: int64


In [127]:
#replace missing with empty string
data_df["subject"] = data_df["subject"].fillna("")
data_df["body"] = data_df["body"].fillna("")

In [128]:
missing_content = (
    (data_df["subject"].str.strip() == "") &
    (data_df["body"].str.strip() == "")
)

print("Emails with no subject or body:", missing_content.sum())

Emails with no subject or body: 617


#### Flagged emails

In [129]:
flagged_df = data_df[missing_content]
flagged_df.head() 
#perhaps make into csv
#may delete these rows, but for now just flag them

,sender,receiver,date,subject,body,label,urls,source_file
91972,有專人為你確認訂購單，所以請安心購買,1,1,,,NaN,NaN,TREC_05.csv
91986,����������.. http://www.freegostop.com,1,1,,,NaN,NaN,TREC_05.csv
92067,下完訂單將有專人再次為你確認，確認無誤後將在48小時內送到你府上 如要訂購 請填寫以...,None,None,,,NaN,NaN,TREC_05.csv
92068,\t\t 聯絡電話: 郵遞區號: 送達時間:,None,None,,,NaN,NaN,TREC_05.csv
92069,\t\t早上,None,None,,,NaN,NaN,TREC_05.csv


#### Decode HTML

Email bodies may contain HTML marks and entities. 

In [130]:
# before decode
def contains_html(text):
    if pd.isna(text):
        return False

    pattern = r"</?(html|body|head|div|p|br|a|span|table|tr|td|img|style|font|meta|title|h[1-6])\b[^>]*>"

    return bool(re.search(pattern, str(text), re.IGNORECASE))


html_emails = data_df[data_df["body"].apply(contains_html)]

print("Emails containing possible HTML:", len(html_emails))

display(html_emails[["subject", "body", "source_file"]].head(10))

Emails containing possible HTML: 36


,subject,body,source_file
23250,Replica Pens,Why would you want to purchase a replica watch...,CEAS_08.csv
72179,Your Apple ID was used to sign in to iMessage ...,Your Apple ID was used to sign in to \t\t\t\t...,Nazario.csv
73361,Fwd: Truly Irish........,\n\nReturn-path: <Comein63@aol.com>\nFrom: Com...,Nazario_5.csv
73374,PanNat Value - Confidential and Proprietary,"Jeff,\n\nPer your request, below is the value ...",Nazario_5.csv
73383,FW: Request for Confidential Information from ...,\n\n -----Original Message-----\nFrom: \tBenev...,Nazario_5.csv
73389,Cougars@Enron - New Interim Dean Named for CBA,"Rod,\nI wanted to forward this to you. Arthur...",Nazario_5.csv
73391,Cougars@Enron UH Alumni Group,Good afternoon.\n\nI wanted to forward you one...,Nazario_5.csv
73485,Your VerticalNet eBusiness Training Services L...,\n<P>\nYour login information for VerticalNet'...,Nazario_5.csv
73500,RE: Americas Gas and Power East doorstep reports,Christy/Shona\nPlease find attached the latest...,Nazario_5.csv
73501,RE: Americas Gas and Power East doorstep reports,"Christy,\nPse find attached the latest review ...",Nazario_5.csv


In [131]:
def clean_email_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities (including nested encoding)
    while True:
        decoded = html.unescape(text)

        if decoded == text:
            break

        text = decoded

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [132]:
data_df["subject"] = data_df["subject"].apply(clean_email_text)
data_df["body"] = data_df["body"].apply(clean_email_text)

In [133]:
data_df['body']

0         Buck up, your troubles caused by small dimensi...
1         Upgrade your sex and pleasures with these tech...
2         >+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...
3         Would anyone object to removing .so from this ...
4         WelcomeFastShippingCustomerSupport http://7iwf...
                                ...                        
218081    While we may have high expectations of our ass...
218082    While we may have high expectations of our ass...
218083    For those who are interested, I just cook a li...
218084    Hello, as I wrote I call sqlFetch(channel,"t_s...
218085    Well, are implicated. When does not the stage ...
Name: body, Length: 218086, dtype: object

In [134]:
#check if html remains
#tags
html_remaining = data_df[
    data_df["body"].apply(contains_html)
]

print("Emails with remaining possible HTML tags:", len(html_remaining))


#html entities
entity_pattern = r"&(?:amp|lt|gt|quot|apos|nbsp);|&#\d+;|&#x[0-9a-fA-F]+;"

entity_remaining = data_df[
    data_df["body"].str.contains(
        entity_pattern,
        case=False,
        regex=True,
        na=False
    )
]

print("Emails with remaining HTML entities:", len(entity_remaining))

Emails with remaining possible HTML tags: 0
Emails with remaining HTML entities: 0


#### New cleaned dataset

In [135]:
#preserve original
print(data_df["source_file"].value_counts())
print("\n\nFinal shape:", data_df.shape)

source_file
TREC_05.csv           55990
TREC_07.csv           53757
CEAS_08.csv           39154
Enron.csv             29767
TREC_06.csv           16457
Nigerian_5.csv         6331
SpamAssasin.csv        5809
Nigerian_Fraud.csv     3332
Nazario_5.csv          3065
Ling.csv               2859
Nazario.csv            1565
Name: count, dtype: int64


Final shape: (218086, 8)


In [136]:
#exporting cleaned dataset
import os

os.makedirs("../data/processed", exist_ok=True)
data_df.to_csv(
    "../data/processed/cleaned_emails.csv",
    index=False
)